# ModelDataset — Master Dataset para XGBoost (World Cup 2026)

Construye el **master dataset de 64 equipos × 24 features** para entrenar el modelo XGBoost.

**Features eliminadas por redundancia**: `ranking_peak`, `wc_goal_diff_per_game`, `log_market_value`.

## Secciones
1. Imports y configuración
2. Carga de datos raw (8 fuentes)
3. Construcción del master dataset (64 × 24)
   - 3.1 Features de ranking FIFA (4)
   - 3.2 Historia mundialista (5)
   - 3.3 Eliminatorias 2026 (4)
   - 3.4 Valores de mercado — Transfermarkt (3)
   - 3.5 Copa América (1)
   - 3.6 Eurocopas (1)
   - 3.7 Win% vs top-10 ponderada (1)
   - 3.8 Merge final + features derivadas (4)
   - 3.9 Verificación y guardado

## Sección 1 — Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 70)
pd.set_option('display.float_format', '{:.3f}'.format)

DATA_DIR = 'Data'

## Sección 2 — Carga de datos raw

In [ ]:
# --- Fuentes principales ---
df_matches   = pd.read_csv(f'{DATA_DIR}/WorldCupHistory/matches_1930_2022.csv')
df_worldcups = pd.read_csv(f'{DATA_DIR}/WorldCupHistory/worldcups.csv')
df_historial = pd.read_csv(f'{DATA_DIR}/WorldCupHistory/historial_mundialista.csv')
df_ranking   = pd.read_csv(f'{DATA_DIR}/ranking_mundial_2026_v2.csv')
df_elim      = pd.read_csv(f'{DATA_DIR}/Eliminatorias/eliminatorias_conjuntas.csv')
df_tm        = pd.read_csv(f'{DATA_DIR}/transfermarkt_selecciones.csv')
df_copa      = pd.read_csv(f'{DATA_DIR}/Copa_America.csv')

# --- 17 archivos de Eurocopas (1960–2024) ---
euro_files = sorted(glob.glob(f'{DATA_DIR}/euro_histoia/*.csv'))
df_euro = pd.concat([pd.read_csv(f) for f in euro_files], ignore_index=True)

print(f"matches:    {df_matches.shape}")
print(f"worldcups:  {df_worldcups.shape}")
print(f"historial:  {df_historial.shape}")
print(f"ranking:    {df_ranking.shape}  | años: {sorted(df_ranking['year'].unique())[:5]}...{sorted(df_ranking['year'].unique())[-3:]}")
print(f"elim:       {df_elim.shape}")
print(f"transfermk: {df_tm.shape}")
print(f"copa:       {df_copa.shape}  | ediciones: {sorted(df_copa['Edição'].unique())}")
print(f"euro:       {df_euro.shape}  | archivos: {len(euro_files)}")

# Lista canónica de 64 equipos (fuente: ranking 2026)
teams_64 = df_ranking[df_ranking['year'] == 2026][['country_code', 'team_name', 'status']].copy()
teams_64 = teams_64.reset_index(drop=True)
print(f"\nEquipos 2026: {len(teams_64)}  (confirmed={teams_64['status'].eq('confirmed').sum()}, playoff={teams_64['status'].eq('playoff').sum()})")

# Confederación de cada equipo (desde eliminatorias)
conf_map = df_elim.set_index('country_code')['confederacion'].to_dict()
# CGO no está en eliminatorias → CAF (Congo es CAF)
conf_map['CGO'] = 'CAF'
teams_64['confederation'] = teams_64['country_code'].map(conf_map)
teams_64['confederation'].value_counts()

## Sección 3 — Construcción del master dataset

### 3.1 — Features de ranking FIFA
`ranking_2026`, `ranking_avg_last5` (2022–2026), `ranking_momentum` (2022−2026), `ranking_volatility`

In [ ]:
LAST5_YEARS = [2022, 2023, 2024, 2025, 2026]

rank_all = df_ranking[df_ranking['year'].isin(LAST5_YEARS)].copy()

def get_ranking_features(code):
    rows = rank_all[rank_all['country_code'] == code].set_index('year')['ranking']

    r2026 = rows.get(2026, np.nan)
    r2022 = rows.get(2022, np.nan)

    available = rows.dropna()
    avg_last5 = available.mean() if len(available) > 0 else np.nan
    volatility = available.std() if len(available) > 1 else 0.0
    momentum = (r2022 - r2026) if (pd.notna(r2022) and pd.notna(r2026)) else 0.0

    return pd.Series({
        'ranking_2026':       r2026,
        'ranking_avg_last5':  avg_last5,
        'ranking_momentum':   momentum,
        'ranking_volatility': volatility,
    })

df_rank_feat = teams_64['country_code'].apply(get_ranking_features)
df_rank_feat.index = teams_64.index

# KVX tiene NaN en todos los años → imputar con el mínimo global del ranking 2026
global_min_ranking = df_ranking[df_ranking['year'] == 2026]['ranking'].max()  # peor ranking = número más alto
kvx_idx = teams_64[teams_64['country_code'] == 'KVX'].index[0]
for col in ['ranking_2026', 'ranking_avg_last5']:
    df_rank_feat.loc[kvx_idx, col] = global_min_ranking

print("Ranking features — spot-check:")
codes_check = ['BRA', 'ARG', 'FRA', 'ESP', 'USA', 'KVX', 'NCL']
idx_check = teams_64[teams_64['country_code'].isin(codes_check)].index
pd.concat([teams_64.loc[idx_check, 'country_code'], df_rank_feat.loc[idx_check]], axis=1)

### 3.2 — Historia mundialista
`wc_win_pct`, `wc_gf_per_game`, `wc_gc_per_game`, `wc_games_played`, `wc_titles`

- **Debutantes** (9 equipos, pj=0): win_pct=0, gf/game=0, gc/game=mediana de su confederación
- **CGO**: no existe en historial → tratado como debutante
- **wc_titles**: conteo desde `worldcups.csv` (incluye Argentina 2022 manualmente)

In [ ]:
# --- Mapeo ganadores worldcups.csv → country_code ---
WINNER_TO_CODE = {
    'Uruguay': 'URU', 'Italy': 'ITA', 'West Germany': 'GER',
    'England': 'ENG', 'Brazil': 'BRA', 'Argentina': 'ARG',
    'Germany': 'GER', 'France': 'FRA', 'Spain': 'ESP',
}

titles_raw = df_worldcups['winner'].map(WINNER_TO_CODE).value_counts()
# Agregar ARG 2022 (worldcups.csv solo llega a 2018)
titles_raw['ARG'] = titles_raw.get('ARG', 0) + 1

# --- Merge historial con la lista de 64 ---
hist = df_historial.set_index('country_code')

# Mediana gc_per_game por confederación (solo equipos con pj > 0)
hist_with_conf = df_historial.merge(
    teams_64[['country_code', 'confederation']], on='country_code', how='inner'
)
gc_median_conf = hist_with_conf[hist_with_conf['pj'] > 0].groupby('confederation')['gc_per_game'].median()

def get_hist_features(row):
    code = row['country_code']
    conf = row['confederation']

    if code in hist.index and hist.loc[code, 'pj'] > 0:
        h = hist.loc[code]
        return pd.Series({
            'wc_games_played':  h['pj'],
            'wc_win_pct':       h['win_pct'],
            'wc_gf_per_game':   h['gf_per_game'],
            'wc_gc_per_game':   h['gc_per_game'],
            'wc_titles':        titles_raw.get(code, 0),
        })
    else:
        # Debutante o CGO (sin historial)
        return pd.Series({
            'wc_games_played':  0,
            'wc_win_pct':       0.0,
            'wc_gf_per_game':   0.0,
            'wc_gc_per_game':   gc_median_conf.get(conf, gc_median_conf.median()),
            'wc_titles':        0,
        })

df_hist_feat = teams_64.apply(get_hist_features, axis=1)

debutantes = teams_64[df_hist_feat['wc_games_played'] == 0]['country_code'].tolist()
print(f"Debutantes ({len(debutantes)}): {debutantes}")
print("\nHistorial features — spot-check:")
codes_check = ['BRA', 'ARG', 'FRA', 'GER', 'CGO', 'UZB', 'ALB']
idx_check = teams_64[teams_64['country_code'].isin(codes_check)].index
pd.concat([teams_64.loc[idx_check, 'country_code'], df_hist_feat.loc[idx_check]], axis=1)

### 3.3 — Eliminatorias 2026
`qual_win_pct`, `qual_gf_per_game`, `qual_gc_per_game`, `qual_points_per_game`

- **USA / MEX / CAN** (hosts, pj=0) → valor del **mejor equipo CONCACAF** por cada feature

In [ ]:
elim = df_elim.copy()
elim['qual_points_per_game'] = np.where(elim['pj'] > 0, elim['pts'] / elim['pj'], 0.0)

# Mejor valor CONCACAF (excluir hosts con pj=0)
concacaf_real = elim[(elim['confederacion'] == 'CONCACAF') & (elim['pj'] > 0)]
best_concacaf = {
    'qual_win_pct':          concacaf_real['win_pct'].max(),
    'qual_gf_per_game':      concacaf_real['gf_per_game'].max(),
    'qual_gc_per_game':      concacaf_real['gc_per_game'].min(),   # menor = mejor
    'qual_points_per_game':  concacaf_real['qual_points_per_game'].max(),
}
print("Mejor CONCACAF por feature:", best_concacaf)

HOST_CODES = {'USA', 'MEX', 'CAN'}
elim_idx = elim.set_index('country_code')

def get_elim_features(row):
    code = row['country_code']
    if code in HOST_CODES:
        return pd.Series(best_concacaf)
    if code not in elim_idx.index:
        # CGO no tiene eliminatorias → mediana de CAF
        caf_elim = elim[elim['confederacion'] == 'CAF']
        return pd.Series({
            'qual_win_pct':         caf_elim['win_pct'].median(),
            'qual_gf_per_game':     caf_elim['gf_per_game'].median(),
            'qual_gc_per_game':     caf_elim['gc_per_game'].median(),
            'qual_points_per_game': (caf_elim['pts'] / caf_elim['pj'].replace(0, np.nan)).median(),
        })
    e = elim_idx.loc[code]
    return pd.Series({
        'qual_win_pct':         e['win_pct'],
        'qual_gf_per_game':     e['gf_per_game'],
        'qual_gc_per_game':     e['gc_per_game'],
        'qual_points_per_game': e['qual_points_per_game'],
    })

df_elim_feat = teams_64.apply(get_elim_features, axis=1)

print("\nEliminatorias features — spot-check:")
codes_check = ['NOR', 'ARG', 'BRA', 'USA', 'MEX', 'CAN', 'CGO', 'CUW']
idx_check = teams_64[teams_64['country_code'].isin(codes_check)].index
pd.concat([teams_64.loc[idx_check, 'country_code'], df_elim_feat.loc[idx_check]], axis=1)

### 3.4 — Valores de mercado (Transfermarkt)
`market_value_eur_m`, `avg_player_value_eur_m`, `squad_avg_age`

- **KOS → KVX** (Kosovo usa código diferente en Transfermarkt)
- **CGO, HAI, CUW, NCL** sin datos → mínimo de su confederación

In [ ]:
tm = df_tm.copy()
# Normalizar código Kosovo
tm['country_code'] = tm['country_code'].replace({'KOS': 'KVX'})
tm_idx = tm.set_index('country_code')

TM_COLS = ['market_value_eur_m', 'avg_player_value_eur_m', 'avg_age']

# Mínimo por confederación (usando conf_map para asignar confederación a TM)
tm['confederation_std'] = tm['country_code'].map(conf_map)
tm_conf_min = (
    tm[tm['market_value_eur_m'].notna()]
    .groupby('confederation_std')[TM_COLS]
    .min()
)

# Mínimos globales como fallback cuando la confederación también tiene NaN
_global_min_mv  = tm['market_value_eur_m'].min()
_global_min_apv = tm['avg_player_value_eur_m'].min()
_global_med_age = tm['avg_age'].median()

def _conf_min(conf, col, fallback):
    """Devuelve el mínimo de confederación o fallback si es NaN/ausente."""
    if conf in tm_conf_min.index:
        v = tm_conf_min.loc[conf, col]
        return v if pd.notna(v) else fallback
    return fallback

def get_tm_features(row):
    code = row['country_code']
    conf = row['confederation']

    if code in tm_idx.index:
        t = tm_idx.loc[code]
        mv  = t['market_value_eur_m']
        apv = t['avg_player_value_eur_m']
        age = t['avg_age']
        # Imputar market_value si falta
        if pd.isna(mv):
            mv  = _conf_min(conf, 'market_value_eur_m',     _global_min_mv)
            apv = _conf_min(conf, 'avg_player_value_eur_m', _global_min_apv)
            age = _conf_min(conf, 'avg_age',                _global_med_age)
        # Imputar avg_player_value si falta (independientemente de mv)
        if pd.isna(apv):
            apv = _conf_min(conf, 'avg_player_value_eur_m', _global_min_apv)
        if pd.isna(age):
            age = _conf_min(conf, 'avg_age', _global_med_age)
        return pd.Series({'market_value_eur_m': mv, 'avg_player_value_eur_m': apv, 'squad_avg_age': age})
    else:
        # Equipo no encontrado → mínimo confederación
        return pd.Series({
            'market_value_eur_m':     _conf_min(conf, 'market_value_eur_m',     _global_min_mv),
            'avg_player_value_eur_m': _conf_min(conf, 'avg_player_value_eur_m', _global_min_apv),
            'squad_avg_age':          _conf_min(conf, 'avg_age',                _global_med_age),
        })

df_tm_feat = teams_64.apply(get_tm_features, axis=1)

print("Transfermarkt features — spot-check:")
codes_check = ['ENG', 'FRA', 'BRA', 'KVX', 'HAI', 'CUW', 'CGO', 'NCL']
idx_check = teams_64[teams_64['country_code'].isin(codes_check)].index
pd.concat([teams_64.loc[idx_check, 'country_code'], df_tm_feat.loc[idx_check]], axis=1)

### 3.5 — Copa América (`copa_win_pct`)
Win% histórica en Copa América. Solo aplica a equipos CONMEBOL; el resto recibe 0.0.

Mapeo de nombres en portugués → country_code.

In [ ]:
COPA_NAME_MAP = {
    'Brasil': 'BRA', 'Argentina': 'ARG', 'Uruguai': 'URU', 'Colombia': 'COL',
    'Equador': 'ECU', 'Paraguai': 'PAR', 'Bolivia': 'BOL', 'Chile': 'CHL',
    'Peru': 'PER', 'Venezuela': 'VEN',
    # Invitados (no CONMEBOL)
    'Honduras': 'HON', 'Costa Rica': 'CRC', 'Mexico': 'MEX',
    'EUA': 'USA', 'Haiti': 'HAI', 'Jamaica': 'JAM',
    'Catar': 'QAT', 'Japao': 'JPN', 'Panama': 'PAN',
}

copa = df_copa.copy()
copa['home_code'] = copa['Casa'].map(COPA_NAME_MAP)
copa['away_code'] = copa['Fora'].map(COPA_NAME_MAP)
copa = copa.dropna(subset=['home_code', 'away_code'])

# Resultado desde perspectiva de cada equipo
records = []
for _, r in copa.iterrows():
    hg, ag = r['Gols Casa'], r['Gols Fora']
    records.append({'code': r['home_code'], 'win': int(hg > ag), 'game': 1})
    records.append({'code': r['away_code'], 'win': int(ag > hg), 'game': 1})

df_copa_stats = pd.DataFrame(records).groupby('code').sum()
df_copa_stats['copa_win_pct'] = df_copa_stats['win'] / df_copa_stats['game']

CONMEBOL_CODES = set(teams_64[teams_64['confederation'] == 'CONMEBOL']['country_code'])

def get_copa_win_pct(code):
    if code not in CONMEBOL_CODES:
        return 0.0
    return df_copa_stats.loc[code, 'copa_win_pct'] if code in df_copa_stats.index else 0.0

df_copa_feat = pd.DataFrame(
    {'copa_win_pct': teams_64['country_code'].map(get_copa_win_pct)}
)

print("Copa América — CONMEBOL teams:")
conmebol_idx = teams_64[teams_64['confederation'] == 'CONMEBOL'].index
pd.concat([teams_64.loc[conmebol_idx, 'country_code'], df_copa_feat.loc[conmebol_idx]], axis=1)

### 3.6 — Eurocopas (`euro_win_pct`)
Win% histórica en Eurocopas (1960–2024). Solo aplica a equipos UEFA; el resto recibe 0.0.

Los archivos ya tienen `home_team_code` / `away_team_code`.

In [ ]:
euro = df_euro[['home_team_code', 'away_team_code', 'home_score', 'away_score']].dropna(
    subset=['home_team_code', 'away_team_code', 'home_score', 'away_score']
).copy()

records_euro = []
for _, r in euro.iterrows():
    hg, ag = r['home_score'], r['away_score']
    records_euro.append({'code': r['home_team_code'], 'win': int(hg > ag), 'game': 1})
    records_euro.append({'code': r['away_team_code'], 'win': int(ag > hg), 'game': 1})

df_euro_stats = pd.DataFrame(records_euro).groupby('code').sum()
df_euro_stats['euro_win_pct'] = df_euro_stats['win'] / df_euro_stats['game']

UEFA_CODES = set(teams_64[teams_64['confederation'] == 'UEFA']['country_code'])

def get_euro_win_pct(code):
    if code not in UEFA_CODES:
        return 0.0
    return df_euro_stats.loc[code, 'euro_win_pct'] if code in df_euro_stats.index else 0.0

df_euro_feat = pd.DataFrame(
    {'euro_win_pct': teams_64['country_code'].map(get_euro_win_pct)}
)

print("Euro — UEFA teams (top 10 por euro_win_pct):")
uefa_idx = teams_64[teams_64['confederation'] == 'UEFA'].index
result_euro = pd.concat([teams_64.loc[uefa_idx, 'country_code'], df_euro_feat.loc[uefa_idx]], axis=1)
result_euro.sort_values('euro_win_pct', ascending=False).head(10)

### 3.7 — Win% vs top-10 ponderada

**Fórmula**: `(2×wc_wins_vs_top10 + cont_wins_vs_top10) / (2×wc_games_vs_top10 + cont_games_vs_top10)`

**Fuentes**:
- WC 2014, 2018, 2022 (peso ×2)
- Euro 2016, 2020, 2024 (solo UEFA)
- Copa América 2016, 2021 (solo CONMEBOL)

**"Top-10"**: rival con ranking ≤ 10 en el año del partido (según `ranking_mundial_2026_v2.csv`)

**Missing** (denominador = 0): mediana de su confederación.

In [ ]:
# ── Años a incluir (todos los que tienen datos de ranking disponibles) ────────
WC_YEARS   = [1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022]
EURO_YEARS = sorted([y for y in df_euro['year'].unique() if y >= 1996])
COPA_YEARS = sorted(df_copa['Edição'].unique().tolist())

# ── Peso temporal: datos más recientes valen más ──────────────────────────────
def time_weight(year):
    if year >= 2019: return 3
    if year >= 2011: return 2
    return 1

# ── Ranking top-10 por año ────────────────────────────────────────────────────
top10_by_year = (
    df_ranking[df_ranking['ranking'] <= 10]
    .groupby('year')['country_code']
    .apply(set)
    .to_dict()
)

# ── Mapeo nombre WC → country_code (cubre todas las ediciones 1994–2022) ─────
WC_NAME_MAP = {
    'Algeria': 'ALG', 'Argentina': 'ARG', 'Australia': 'AUS', 'Austria': 'AUT',
    'Belgium': 'BEL', 'Bolivia': 'BOL', 'Bosnia and Herzegovina': 'BIH',
    'Brazil': 'BRA', 'Canada': 'CAN', 'Chile': 'CHL', 'Colombia': 'COL',
    'Costa Rica': 'CRC', 'Croatia': 'CRO', "Côte d'Ivoire": 'CIV',
    'Czech Republic': 'CZE', 'Denmark': 'DEN', 'Ecuador': 'ECU', 'Egypt': 'EGY',
    'England': 'ENG', 'France': 'FRA', 'Germany': 'GER', 'Ghana': 'GHA',
    'Greece': 'GRE', 'Honduras': 'HON', 'IR Iran': 'IRN', 'Iceland': 'ISL',
    'Italy': 'ITA', 'Jamaica': 'JAM', 'Japan': 'JPN', 'Korea Republic': 'KOR',
    'Mexico': 'MEX', 'Morocco': 'MAR', 'Netherlands': 'NED', 'New Zealand': 'NZL',
    'Nigeria': 'NGA', 'Norway': 'NOR', 'Panama': 'PAN', 'Paraguay': 'PAR',
    'Peru': 'PER', 'Poland': 'POL', 'Portugal': 'POR', 'Qatar': 'QAT',
    'Republic of Ireland': 'IRL', 'Romania': 'ROU', 'Russia': 'RUS',
    'Saudi Arabia': 'KSA', 'Scotland': 'SCO', 'Senegal': 'SEN', 'Serbia': 'SRB',
    'Slovakia': 'SVK', 'South Africa': 'RSA', 'Spain': 'ESP', 'Sweden': 'SWE',
    'Switzerland': 'SUI', 'Tunisia': 'TUN', 'Türkiye': 'TUR', 'Ukraine': 'UKR',
    'United States': 'USA', 'Uruguay': 'URU', 'Wales': 'WAL',
}

# ── Acumuladores ponderados ───────────────────────────────────────────────────
from collections import defaultdict

wc_wins_w  = defaultdict(float); wc_games_w  = defaultdict(float)
cont_wins_w = defaultdict(float); cont_games_w = defaultdict(float)

# -- Mundiales 1994–2022 (peso_fuente=2 × peso_tiempo) ------------------------
wc_data = df_matches[df_matches['Year'].isin(WC_YEARS)].copy()

for _, r in wc_data.iterrows():
    year = int(r['Year'])
    hc = WC_NAME_MAP.get(r['home_team'])
    ac = WC_NAME_MAP.get(r['away_team'])
    if hc is None or ac is None:
        continue
    top10 = top10_by_year.get(year, set())
    hg, ag = r['home_score'], r['away_score']
    if pd.isna(hg) or pd.isna(ag):
        continue
    hg, ag = int(hg), int(ag)
    tw = time_weight(year)

    if ac in top10:
        wc_games_w[hc] += tw
        wc_wins_w[hc]  += tw * int(hg > ag)
    if hc in top10:
        wc_games_w[ac] += tw
        wc_wins_w[ac]  += tw * int(ag > hg)

# -- Eurocopas desde 1996 (peso_fuente=1 × peso_tiempo) -----------------------
euro_cont = df_euro[df_euro['year'].isin(EURO_YEARS)].copy()
euro_cont = euro_cont.dropna(subset=['home_team_code', 'away_team_code', 'home_score', 'away_score'])

for _, r in euro_cont.iterrows():
    year = int(r['year'])
    hc, ac = r['home_team_code'], r['away_team_code']
    top10 = top10_by_year.get(year, set())
    hg, ag = int(r['home_score']), int(r['away_score'])
    tw = time_weight(year)

    if ac in top10:
        cont_games_w[hc] += tw
        cont_wins_w[hc]  += tw * int(hg > ag)
    if hc in top10:
        cont_games_w[ac] += tw
        cont_wins_w[ac]  += tw * int(ag > hg)

# -- Copa América (todas las ediciones disponibles, peso_fuente=1 × peso_tiempo)
copa_cont = df_copa[df_copa['Edição'].isin(COPA_YEARS)].copy()
copa_cont['home_code'] = copa_cont['Casa'].map(COPA_NAME_MAP)
copa_cont['away_code'] = copa_cont['Fora'].map(COPA_NAME_MAP)
copa_cont = copa_cont.dropna(subset=['home_code', 'away_code'])

for _, r in copa_cont.iterrows():
    year = int(r['Edição'])
    hc, ac = r['home_code'], r['away_code']
    top10 = top10_by_year.get(year, set())
    hg, ag = r['Gols Casa'], r['Gols Fora']
    if pd.isna(hg) or pd.isna(ag):
        continue
    hg, ag = int(hg), int(ag)
    tw = time_weight(year)

    if ac in top10:
        cont_games_w[hc] += tw
        cont_wins_w[hc]  += tw * int(hg > ag)
    if hc in top10:
        cont_games_w[ac] += tw
        cont_wins_w[ac]  += tw * int(ag > hg)

# ── Calcular win_pct_vs_top10 ponderado ───────────────────────────────────────
def calc_top10_pct(code):
    # peso_fuente: WC=2, continental=1
    w = 2 * wc_wins_w[code]  + cont_wins_w[code]
    g = 2 * wc_games_w[code] + cont_games_w[code]
    if g == 0:
        return np.nan
    return w / g

raw_top10 = teams_64['country_code'].map(calc_top10_pct)

# Imputar sin datos: mediana confederación, luego mediana global
teams_64['_top10_raw'] = raw_top10.values
conf_median_top10 = teams_64.groupby('confederation')['_top10_raw'].median()
_global_median_top10 = teams_64['_top10_raw'].median()

def fill_top10(row):
    v = row['_top10_raw']
    if pd.isna(v):
        conf_med = conf_median_top10.get(row['confederation'])
        return conf_med if pd.notna(conf_med) else _global_median_top10
    return v

df_top10_feat = pd.DataFrame({
    'win_pct_vs_top10': teams_64.apply(fill_top10, axis=1).values
}, index=teams_64.index)

teams_64.drop(columns=['_top10_raw'], inplace=True)

print(f"WC editions: {WC_YEARS}")
print(f"Euro editions: {EURO_YEARS}")
print(f"Copa editions: {COPA_YEARS}")
print("\nWin% vs top-10 (ponderado) — top 10 equipos:")
result_top10 = pd.concat([teams_64[['country_code', 'confederation']], df_top10_feat], axis=1)
result_top10.sort_values('win_pct_vs_top10', ascending=False).head(10)

### 3.8 — Merge final + features derivadas
`wc_debut_flag`, `host_flag`, `is_playoff`, `confederation_strength_index`

In [ ]:
# ── Merge de todos los bloques de features ───────────────────────────────────
df_master = pd.concat([
    teams_64[['country_code', 'confederation', 'status']].reset_index(drop=True),
    df_rank_feat.reset_index(drop=True),
    df_hist_feat.reset_index(drop=True),
    df_elim_feat.reset_index(drop=True),
    df_tm_feat.reset_index(drop=True),
    df_copa_feat.reset_index(drop=True),
    df_euro_feat.reset_index(drop=True),
    df_top10_feat.reset_index(drop=True),
], axis=1)

# ── Features derivadas ────────────────────────────────────────────────────────
df_master['wc_debut_flag'] = (df_master['wc_games_played'] == 0).astype(int)
df_master['host_flag']     = df_master['country_code'].isin(HOST_CODES).astype(int)
df_master['is_playoff']    = (df_master['status'] == 'playoff').astype(int)

conf_avg_ranking = df_master.groupby('confederation')['ranking_2026'].mean()
df_master['confederation_strength_index'] = (
    1.0 / df_master['confederation'].map(conf_avg_ranking)
)

# ── Orden final de columnas (16 features) ─────────────────────────────────────
FEATURE_COLS = [
    # Ranking (2) — eliminado ranking_avg_last5 (corr=0.99), ranking_momentum
    'ranking_2026', 'ranking_volatility',
    # Historia WC (3) — eliminado wc_gf_per_game (corr=0.94) y wc_games_played (corr=0.76–0.81)
    'wc_win_pct', 'wc_gc_per_game', 'wc_titles',
    # Eliminatorias (3) — eliminado qual_win_pct (corr=0.97 con qual_points_per_game)
    'qual_gf_per_game', 'qual_gc_per_game', 'qual_points_per_game',
    # Mercado (2) — eliminado avg_player_value_eur_m (corr=0.99 con market_value_eur_m)
    'market_value_eur_m', 'squad_avg_age',
    # Torneos continentales (2)
    'copa_win_pct', 'euro_win_pct',
    # vs top-10 ponderado (1)
    'win_pct_vs_top10',
    # Derivadas (4)
    'wc_debut_flag', 'host_flag', 'is_playoff', 'confederation_strength_index',
]

df_master = df_master[['country_code', 'confederation', 'status'] + FEATURE_COLS]

print(f"Shape: {df_master.shape}")
print(f"Features: {len(FEATURE_COLS)}")
df_master.head()

### 3.9 — Verificación y guardado

In [ ]:
# ── Assert 1: exactamente 64 equipos ─────────────────────────────────────────
assert len(df_master) == 64, f"ERROR: {len(df_master)} filas (esperado 64)"
print(f"✓ 64 equipos")

# ── Assert 2: cero NaN en las 18 features ────────────────────────────────────
nan_counts = df_master[FEATURE_COLS].isna().sum()
nan_cols = nan_counts[nan_counts > 0]
assert len(nan_cols) == 0, f"ERROR: NaN en columnas:\n{nan_cols}"
print(f"✓ Sin NaN en las {len(FEATURE_COLS)} features")

# ── Spot-checks ───────────────────────────────────────────────────────────────
bra = df_master[df_master['country_code'] == 'BRA'].iloc[0]
assert bra['ranking_2026'] == 5,         f"BRA ranking esperado 5, obtenido {bra['ranking_2026']}"
assert abs(bra['wc_win_pct'] - 0.667) < 0.01, f"BRA wc_win_pct inesperado: {bra['wc_win_pct']}"
assert abs(bra['market_value_eur_m'] - 932) < 1, f"BRA market_value inesperado: {bra['market_value_eur_m']}"
print(f"✓ BRA: ranking={bra['ranking_2026']}, wc_win_pct={bra['wc_win_pct']:.3f}, market={bra['market_value_eur_m']}M")

usa = df_master[df_master['country_code'] == 'USA'].iloc[0]
assert usa['host_flag'] == 1, "USA host_flag debe ser 1"
assert usa['qual_points_per_game'] > 0, "USA qual_points_per_game debe ser > 0"
print(f"✓ USA: host_flag={usa['host_flag']}, qual_ppg={usa['qual_points_per_game']:.3f}")

debut_teams = df_master[df_master['wc_debut_flag'] == 1]['country_code'].tolist()
print(f"✓ Debutantes ({len(debut_teams)}): {sorted(debut_teams)}")

# ── Top-10 win_pct_vs_top10 ───────────────────────────────────────────────────
print("\nTop 10 equipos por win_pct_vs_top10 (ponderado):")
print(df_master[['country_code', 'win_pct_vs_top10']].sort_values('win_pct_vs_top10', ascending=False).head(10).to_string(index=False))

# ── Guardar ───────────────────────────────────────────────────────────────────
out_path = 'Data/master_dataset_64.csv'
df_master.to_csv(out_path, index=False)
print(f"\n✓ Guardado: {out_path}  ({df_master.shape[0]} filas × {df_master.shape[1]} columnas)")